# AUAF — Stage 4.8: Final Reliability, Calibration and Conformal Analysis

This is the final local-execution notebook for Stage 4.8. It uses the frozen Stage 4.4 temporal-clean datasets and the already-trained Stage 4.5 neural/tree checkpoints. It does **not** retrain models, retune hyperparameters, change participant splits, or modify prior-stage artifacts.

All ADNI participant-level data, predictions, calibration scores, prediction sets, and checkpoints remain inside the user's institution-approved computational environment. The aggregate-only release generated at the end excludes row-level files.


## 1. Local execution and data-use requirements

1. This execution uses the user-confirmed authorization for ADNI processing in the current ChatGPT analysis workspace.
2. The frozen Stage 4.7 data package supplies the Stage 4.4 as-of-index cohorts and split manifests; the final Stage 4.5 package supplies the frozen checkpoints.
3. Participant-level outputs remain inside this analysis workspace and are excluded from the aggregate-only release ZIP. No checkpoint is retrained and no cohort is resplit.
4. Set `AUAF_STAGE45_ROOT` if the Stage 4.5 extraction folder differs from the configured path.
5. Run all cells from top to bottom. The notebook performs inference and statistical post-processing only.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Mapping, Sequence, Any
import hashlib, json, platform, sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT_DIR = Path.cwd()
STAGE45_ROOT = Path(__import__("os").environ.get(
    "AUAF_STAGE45_ROOT", str(PROJECT_DIR / "AUAF_Stage4_5_MissingModality_FinalCohort")
)).expanduser().resolve()
DATA_DIR = PROJECT_DIR / "stage4_data"
REFERENCE_DIR = DATA_DIR / "reference_splits"
OUTPUT_ROOT = PROJECT_DIR / "outputs"
RUN_ID = "stage4_8_preflight_authorized_v2"
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID

N_BOOT = 2000
LATENT_DIM = 128
MODALITY_DROPOUT = 0.20
MC_PASSES = 30
MODEL_VARIANTS = ["EarlyFusion", "LateFusion", "AttentionFusion", "AUAF"]

print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Output:", OUTPUT_DIR)
print("PyTorch:", torch.__version__)


## 2. Verify the frozen Stage 4.4 inputs

The four dataset files are checked against the SHA-256 values recorded for the final temporal-clean package. A mismatch stops execution. The cohort definitions and frozen participant-level splits are not regenerated.


In [ ]:
EXPECTED_SHA256 = {'diagnosis_binary_3700_asof.csv': '0d267c6510f78a42207df7242f6027a413da220c05c43d8268c9f17a24d3810a', 'diagnosis_multiclass_3700_asof.csv': 'aa17acdc7048d5a83d5e38b93beeea65cbe5a26560df0d8258c409f0f030f4a1', 'feature_manifest_34_predictors.csv': 'feb1cb2d61606544045339f2a34bc32a9f8cc786dc1aaf2a3893539e011e5184', 'mci_conversion_911_asof.csv': 'd95430f010c8e5a5d40d3e5a69038cfa6c7fbab736a26dd4cc78ced00648f9ad'}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

missing_files = [name for name in EXPECTED_SHA256 if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing required inputs: {missing_files}")

ACTUAL_SHA256 = {name: sha256_file(DATA_DIR / name) for name in EXPECTED_SHA256}
for name, expected in EXPECTED_SHA256.items():
    if ACTUAL_SHA256[name] != expected:
        raise RuntimeError(f"SHA-256 mismatch: {name}")

print("Input SHA-256 verification: PASS")


## 3. Verify seeds and frozen split assignments

The five values `[42, 123, 456, 789, 1024]` are model-initialization seeds on the same subject-disjoint train/validation/test split. They are **not** five independent folds or cohorts.


In [ ]:
stage45_run_manifest = json.loads((STAGE45_ROOT / "neural_training_run_manifest.json").read_text(encoding="utf-8"))
stage45_manifest = json.loads((STAGE45_ROOT / "STAGE4_5_MANIFEST.json").read_text(encoding="utf-8"))
SEEDS = [int(x) for x in stage45_run_manifest["immutable_manifest"]["seeds"]]
assert SEEDS == [42, 123, 456, 789, 1024]
assert SEEDS == [int(x) for x in stage45_manifest["neural_models"]["seeds"]]
assert SEEDS == [int(x) for x in stage45_manifest["tree_baselines"]["seeds"]]
assert stage45_manifest["neural_models"]["jobs"] == 60
assert stage45_manifest["tree_baselines"]["jobs"] == 45

SPLIT_REFERENCE = {
    "diagnosis_multiclass": ("subject_split_manifest__cn_mci_ad.csv", "target_multiclass", "TARGET_MULTICLASS"),
    "diagnosis_binary": ("subject_split_manifest__ad_vs_nonad.csv", "target_binary", "TARGET_BINARY"),
    "mci_conversion": ("subject_split_manifest__early_decline.csv", "target_decline", "TARGET_CONVERSION"),
}

print("Verified five seeds:", SEEDS)
print("Seed interpretation: model initialization/tuning seeds; not independent outer splits.")


## 4. Load the three final task cohorts

The notebook uses the final temporal-clean Stage 4.4 cohorts: 3,700 participants for each diagnostic task and 911 participants for 36-month MCI-to-AD conversion.


In [ ]:
TASK_FILES = {
    "diagnosis_multiclass": ("diagnosis_multiclass_3700_asof.csv", "TARGET_MULTICLASS", 3700, 3),
    "diagnosis_binary": ("diagnosis_binary_3700_asof.csv", "TARGET_BINARY", 3700, 2),
    "mci_conversion": ("mci_conversion_911_asof.csv", "TARGET_CONVERSION", 911, 2),
}

TASKS = {}
for task_name, (filename, target_col, expected_n, n_classes) in TASK_FILES.items():
    frame = pd.read_csv(DATA_DIR / filename, parse_dates=["INDEX_DATE"])
    if len(frame) != expected_n:
        raise AssertionError(f"{task_name}: expected {expected_n} rows; found {len(frame)}")
    TASKS[task_name] = {
        "frame": frame,
        "target": target_col,
        "n_classes": n_classes,
        "filename": filename,
    }

FEATURE_MANIFEST = pd.read_csv(DATA_DIR / "feature_manifest_34_predictors.csv")
if FEATURE_MANIFEST["feature"].duplicated().any():
    raise AssertionError("Duplicate feature names in feature manifest.")

FEATURES = FEATURE_MANIFEST["feature"].astype(str).tolist()
CLINICAL_FEATURES = FEATURE_MANIFEST.loc[
    FEATURE_MANIFEST["modality"].eq("clinical_cognitive"), "feature"
].astype(str).tolist()
BIOMARKER_FEATURES = FEATURE_MANIFEST.loc[
    FEATURE_MANIFEST["modality"].eq("fluid_biomarker"), "feature"
].astype(str).tolist()

assert len(FEATURES) == 34
assert len(CLINICAL_FEATURES) == 31
assert len(BIOMARKER_FEATURES) == 3
assert set(CLINICAL_FEATURES).isdisjoint(BIOMARKER_FEATURES)
assert set(CLINICAL_FEATURES) | set(BIOMARKER_FEATURES) == set(FEATURES)
assert not ({"LANGUAGE_CODE", "APVOLUME"} & set(FEATURES))

REQUIRED_META = ["RID", "split", "INDEX_DATE", "BIOMARKER_AFTER_INDEX"]
ALLOWED_SPLITS = {"train", "validation", "test"}
EXPECTED_MISSING_INDEX_DATES = {
    "diagnosis_multiclass": 15, "diagnosis_binary": 15, "mci_conversion": 0
}

def validate_task(task_name: str, spec: dict) -> dict:
    frame, target = spec["frame"], spec["target"]
    required = set(REQUIRED_META + [target] + FEATURES)
    missing = sorted(required - set(frame.columns))
    if missing:
        raise AssertionError(f"{task_name}: missing columns {missing}")
    if frame["RID"].isna().any() or frame["RID"].duplicated().any():
        raise AssertionError(f"{task_name}: RID must be nonmissing and unique.")
    missing_index_dates = int(frame["INDEX_DATE"].isna().sum())
    if missing_index_dates != EXPECTED_MISSING_INDEX_DATES[task_name] or frame[target].isna().any():
        raise AssertionError(
            f"{task_name}: expected {EXPECTED_MISSING_INDEX_DATES[task_name]} missing index dates; "
            f"observed {missing_index_dates}; target missing={bool(frame[target].isna().any())}."
        )
    if set(frame["split"].astype(str).str.lower().unique()) != ALLOWED_SPLITS:
        raise AssertionError(f"{task_name}: unexpected split labels.")
    if frame[FEATURES].isin([-1, -4]).to_numpy().any():
        raise AssertionError(f"{task_name}: ADNI missing sentinels remain.")
    if not all(pd.api.types.is_numeric_dtype(frame[c]) for c in FEATURES):
        raise AssertionError(f"{task_name}: expected numeric predictors only.")
    observed_any = frame[FEATURES].notna().any(axis=1)
    if not observed_any.all():
        raise AssertionError(f"{task_name}: participant with no available model predictor detected.")
    if frame.loc[frame["BIOMARKER_AFTER_INDEX"].astype(bool), BIOMARKER_FEATURES].notna().any(axis=None):
        raise AssertionError(f"{task_name}: post-index biomarker values survived the as-of data build.")
    if frame.loc[frame["INDEX_DATE"].isna(), BIOMARKER_FEATURES].notna().any(axis=None):
        raise AssertionError(f"{task_name}: biomarker values present when the index date is missing.")
    split_sets = {
        s: set(frame.loc[frame["split"].astype(str).str.lower().eq(s), "RID"])
        for s in ALLOWED_SPLITS
    }
    for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
        if split_sets[a] & split_sets[b]:
            raise AssertionError(f"{task_name}: participant overlap across {a}/{b}.")
    return {
        "task": task_name,
        "n": int(len(frame)),
        "class_counts": {str(k): int(v) for k, v in frame[target].value_counts().sort_index().items()},
        "split_counts": {str(k): int(v) for k, v in frame["split"].value_counts().items()},
        "unique_RID": int(frame["RID"].nunique()),
        "all_predictors_numeric": True,
        "no_ADNI_sentinels": True,
    }

VALIDATION_SUMMARY = pd.DataFrame([
    validate_task(name, spec) for name, spec in TASKS.items()
])
print(VALIDATION_SUMMARY.to_string(index=False))


## 5. Exact comparison with the archived split manifests

A mismatch in any participant, split label, or target label stops the notebook. No split is regenerated.


In [ ]:
split_check_rows = []
for task_name, (reference_name, reference_target, current_target) in SPLIT_REFERENCE.items():
    ref_path = REFERENCE_DIR / reference_name
    if not ref_path.is_file():
        raise FileNotFoundError(ref_path)
    ref = pd.read_csv(ref_path).rename(columns={"_split": "split"})
    cur = TASKS[task_name]["frame"][["RID", "split", current_target]].copy()
    merged = ref[["RID", "split", reference_target]].merge(
        cur, on="RID", how="outer", suffixes=("_archived", "_current"),
        indicator=True, validate="one_to_one"
    )
    if not merged["_merge"].eq("both").all():
        raise AssertionError(f"{task_name}: participant roster differs from archived split manifest.")
    split_match = merged["split_archived"].astype(str).str.lower().eq(
        merged["split_current"].astype(str).str.lower()
    )
    target_match = pd.to_numeric(merged[reference_target], errors="raise").eq(
        pd.to_numeric(merged[current_target], errors="raise")
    )
    if not split_match.all():
        raise AssertionError(f"{task_name}: split assignments differ from archived manifest.")
    if not target_match.all():
        raise AssertionError(f"{task_name}: target labels differ from archived manifest.")
    split_check_rows.append({
        "task": task_name,
        "participants_checked": int(len(merged)),
        "RID_match_pct": 100.0,
        "split_match_pct": 100.0,
        "target_match_pct": 100.0,
        "status": "PASS",
    })

SPLIT_CHECK = pd.DataFrame(split_check_rows)
print(SPLIT_CHECK.to_string(index=False))


## 6. Verified predictors and prespecified feature sets

The primary analysis uses `S0_verified_asof`: 29 verified clinical/cognitive predictors and three fluid biomarkers. `PTENGSPKAGE` and `PTIMMAGE` remain excluded because their phase-specific definitions were not verified. The seven feature sets are retained for provenance and sensitivity planning; Stage 4.8 inference below uses S0 to match the final Stage 4.5 checkpoints.


In [ ]:
UNVERIFIED_FIELDS = ["PTENGSPKAGE", "PTIMMAGE"]
VERIFIED_FEATURES = [f for f in FEATURES if f not in UNVERIFIED_FIELDS]

DEMOGRAPHIC = ["PTGENDER", "PTEDUCAT"]
MMSE = ["MMSEASON"]
MOCA = ["TRAILS", "CLOCKCON", "CLOCKNO", "CLOCKHAN", "FFLUENCY", "MOCA"]
FAQ = ["FAQFINAN", "FAQFORM", "FAQSHOP", "FAQGAME", "FAQBEVG", "FAQMEAL",
       "FAQEVENT", "FAQTV", "FAQREM", "FAQTRAVL", "FAQTOTAL"]
CDR = ["CDMEMORY", "CDRSB"]
NEUROBAT = ["CLOCKCIRC", "CLOCKSYM", "CLOCKNUM", "CLOCKHAND", "CLOCKTIME", "CLOCKSCOR", "DIGITSCOR"]
BIOMARKERS = ["ABETA40", "ABETA42", "PTAU"]

assert len(FEATURES) == 34
assert len(VERIFIED_FEATURES) == 32
assert len(CLINICAL_FEATURES) == 31
assert len(BIOMARKER_FEATURES) == 3
assert len([f for f in VERIFIED_FEATURES if f in CLINICAL_FEATURES]) == 29
assert set(DEMOGRAPHIC + MMSE + MOCA + FAQ + CDR + NEUROBAT + BIOMARKERS) == set(VERIFIED_FEATURES)

FEATURE_SETS = {
    "S0_verified_asof": VERIFIED_FEATURES.copy(),
    "S1_minus_CDR": [f for f in VERIFIED_FEATURES if f not in CDR],
    "S2_minus_FAQ": [f for f in VERIFIED_FEATURES if f not in FAQ],
    "S3_minus_CDR_FAQ": [f for f in VERIFIED_FEATURES if f not in CDR + FAQ],
    "S4_minus_MMSE_MOCA": [f for f in VERIFIED_FEATURES if f not in MMSE + MOCA],
    "S5_strict_CDR_FAQ_MMSE_MOCA": [
        f for f in VERIFIED_FEATURES if f not in CDR + FAQ + MMSE + MOCA
    ],
    "S6_demographic_biomarker_only": DEMOGRAPHIC + BIOMARKERS,
}

FEATURE_SET_TABLE = pd.DataFrame([
    {
        "feature_set": name,
        "candidate_predictor_count": len(cols),
        "removed_features": ";".join([f for f in FEATURES if f not in cols]),
        "retained_features": ";".join(cols),
        "analysis_role": (
            "Exploratory reference for conversion; not evidence of multimodal fusion"
            if name == "S6_demographic_biomarker_only"
            else (
                "Primary verified as-of-index candidate"
                if name == "S0_verified_asof"
                else "Prespecified circularity/severity sensitivity set"
            )
        ),
        "status": "Candidate — not yet trained",
    }
    for name, cols in FEATURE_SETS.items()
])

definition_hold = pd.DataFrame({
    "feature": UNVERIFIED_FIELDS,
    "action": ["Excluded from all Stage 4.8 model feature sets pending source-definition verification"] * len(UNVERIFIED_FIELDS),
})

print(FEATURE_SET_TABLE[["feature_set", "candidate_predictor_count", "analysis_role"]].to_string(index=False))
print("\nUnverified ADNI fields excluded from all model feature sets:")
print(definition_hold.to_string(index=False))


## 7. Train-only preprocessing reconstruction

Preprocessing objects are reconstructed from the frozen Stage 4.4 table and fitted on the training partition only. They are used only to prepare the same model inputs for inference with existing checkpoints; no preprocessing is fitted on validation or test participants.


In [ ]:
class TrainOnlyNumericPreprocessor:
    def __init__(self, columns: Sequence[str]):
        self.columns = list(columns)
        self.medians_ = None
        self.means_ = None
        self.scales_ = None
        self.all_missing_train_ = None
        self.fit_rids_ = None
        self.fit_n_ = None

    @staticmethod
    def _array(frame: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        values = frame[list(columns)].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)
        values[~np.isfinite(values)] = np.nan
        return values

    def fit(self, frame: pd.DataFrame, train_mask: np.ndarray):
        train_mask = np.asarray(train_mask, dtype=bool)
        if train_mask.sum() == 0:
            raise ValueError("Cannot fit preprocessing: empty training split.")
        x_train = self._array(frame.loc[train_mask], self.columns)

        medians = np.zeros(x_train.shape[1], dtype=np.float64)
        all_missing = np.zeros(x_train.shape[1], dtype=bool)
        for j in range(x_train.shape[1]):
            observed = x_train[:, j][np.isfinite(x_train[:, j])]
            if observed.size == 0:
                all_missing[j] = True
                medians[j] = 0.0
            else:
                medians[j] = float(np.median(observed))

        # Impute with training medians.
        x_imp = np.where(np.isfinite(x_train), x_train, medians[None, :])
        means = x_imp.mean(axis=0)
        scales = x_imp.std(axis=0, ddof=0)
        scales[~np.isfinite(scales) | (scales < 1e-12)] = 1.0

        self.medians_ = medians
        self.means_ = means
        self.scales_ = scales
        self.all_missing_train_ = all_missing
        self.fit_rids_ = sorted(frame.loc[train_mask, "RID"].astype(int).tolist())
        self.fit_n_ = int(train_mask.sum())
        return self

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        if self.medians_ is None:
            raise RuntimeError("Preprocessor has not been fitted.")
        x = self._array(frame, self.columns)
        # Do not allow features absent in training to contribute test-only information.
        x[:, self.all_missing_train_] = np.nan
        x_imp = np.where(np.isfinite(x), x, self.medians_[None, :])
        z = (x_imp - self.means_[None, :]) / self.scales_[None, :]
        z[~np.isfinite(z)] = 0.0
        return z.astype(np.float32)

    def effective_columns(self) -> List[str]:
        return [c for c, missing in zip(self.columns, self.all_missing_train_) if not missing]

    def audit(self) -> dict:
        return {
            "columns_requested": self.columns,
            "columns_effective": self.effective_columns(),
            "all_missing_in_train": [
                c for c, missing in zip(self.columns, self.all_missing_train_) if missing
            ],
            "fit_n": self.fit_n_,
            "fit_RID_count": len(self.fit_rids_),
            "fit_RID_sha256": hashlib.sha256(
                ",".join(map(str, self.fit_rids_)).encode("utf-8")
            ).hexdigest(),
        }


@dataclass
class PreparedTask:
    name: str
    feature_set: str
    frame: pd.DataFrame
    target: str
    n_classes: int
    modalities: List[str]
    features_by_modality: Dict[str, List[str]]
    matrices: Dict[str, np.ndarray]
    masks: np.ndarray
    processors: Dict[str, TrainOnlyNumericPreprocessor]
    fit_rids: List[int]

    def idx(self, split: str, eligible_only: bool = False) -> np.ndarray:
        split_mask = self.frame["split"].astype(str).str.lower().eq(split).to_numpy()
        idx = np.flatnonzero(split_mask)
        if eligible_only:
            idx = idx[self.masks[idx].sum(axis=1) > 0]
        return idx

    def paired_modality_idx(self, split: str) -> np.ndarray:
        idx = self.idx(split, eligible_only=False)
        if self.masks.shape[1] < 2:
            return np.array([], dtype=int)
        return idx[(self.masks[idx] > 0).all(axis=1)]


MODALITY_ORDER = ["clinical_cognitive", "fluid_biomarker"]

def prepare_task(task_name: str, feature_set_name: str) -> PreparedTask:
    spec = TASKS[task_name]
    frame = spec["frame"].reset_index(drop=True).copy()
    selected = FEATURE_SETS[feature_set_name]
    train_mask = frame["split"].astype(str).str.lower().eq("train").to_numpy()

    features_by_modality = {}
    matrices = {}
    processors = {}
    mask_columns = []
    active_modalities = []
    all_fit_rids = None

    for modality in MODALITY_ORDER:
        requested = [
            f for f in selected
            if f in set(FEATURE_MANIFEST.loc[
                FEATURE_MANIFEST["modality"].eq(modality), "feature"
            ].astype(str))
        ]
        if not requested:
            continue

        processor = TrainOnlyNumericPreprocessor(requested).fit(frame, train_mask)
        effective = processor.effective_columns()
        if not effective:
            continue

        # Refit on effective columns only, still using training participants only.
        processor = TrainOnlyNumericPreprocessor(effective).fit(frame, train_mask)
        values = processor.transform(frame)
        raw_available = frame[effective].notna().any(axis=1).to_numpy()

        features_by_modality[modality] = effective
        matrices[modality] = values
        processors[modality] = processor
        mask_columns.append(raw_available.astype(np.float32))
        active_modalities.append(modality)

        current_fit_rids = processor.fit_rids_
        if all_fit_rids is None:
            all_fit_rids = current_fit_rids
        elif current_fit_rids != all_fit_rids:
            raise AssertionError("Preprocessors must be fitted on the same training participants.")

    if not active_modalities:
        raise RuntimeError(f"{task_name}/{feature_set_name}: no train-usable modality.")

    masks = np.column_stack(mask_columns).astype(np.float32)
    if not np.isfinite(masks).all():
        raise AssertionError("Non-finite modality mask.")
    for modality, values in matrices.items():
        if values.shape[0] != len(frame) or not np.isfinite(values).all():
            raise AssertionError(f"Invalid transformed matrix for {task_name}/{feature_set_name}/{modality}.")

    return PreparedTask(
        name=task_name,
        feature_set=feature_set_name,
        frame=frame,
        target=spec["target"],
        n_classes=spec["n_classes"],
        modalities=active_modalities,
        features_by_modality=features_by_modality,
        matrices=matrices,
        masks=masks,
        processors=processors,
        fit_rids=all_fit_rids or [],
    )


PREPARED = {
    (task_name, feature_set_name): prepare_task(task_name, feature_set_name)
    for task_name in TASKS
    for feature_set_name in FEATURE_SETS
}
print("Prepared task/feature-set combinations:", len(PREPARED))

preprocessor_fit_rows = []
for (task_name, feature_set_name), pack in PREPARED.items():
    expected_train_rids = set(
        pack.frame.loc[
            pack.frame["split"].astype(str).str.lower().eq("train"), "RID"
        ].astype(int)
    )
    for modality, processor in pack.processors.items():
        fitted_rids = set(processor.fit_rids_)
        if fitted_rids != expected_train_rids:
            raise AssertionError(
                f"{task_name}/{feature_set_name}/{modality}: preprocessor fit IDs are not exactly train IDs."
            )
        preprocessor_fit_rows.append({
            "task": task_name,
            "feature_set": feature_set_name,
            "modality": modality,
            "fit_participant_n": processor.fit_n_,
            "expected_train_participant_n": len(expected_train_rids),
            "fit_RID_exactly_train_RID": True,
            "all_missing_in_train_features": ";".join(processor.audit()["all_missing_in_train"]),
            "effective_feature_n": len(processor.effective_columns()),
        })

PREPROCESSOR_FIT_AUDIT = pd.DataFrame(preprocessor_fit_rows)
print("Train-only preprocessing fit audit: PASS", len(PREPROCESSOR_FIT_AUDIT), "modality pipelines")



## 8. Participant availability and matched-cohort audit

The same frozen participant rows and labels are retained across all model comparisons. Complete-case modality analyses use the matched test participants with both clinical/cognitive and fluid-biomarker modalities available at baseline.


In [ ]:
eligibility_rows = []
matched_rows = []
overlap_rows = []

for task_name, spec in TASKS.items():
    frame = spec["frame"].reset_index(drop=True)
    target = spec["target"]
    prepared_s0 = PREPARED[(task_name, "S0_verified_asof")]

    for feature_set_name in FEATURE_SETS:
        pack = PREPARED[(task_name, feature_set_name)]
        for split in ["train", "validation", "test"]:
            all_idx = pack.idx(split, eligible_only=False)
            eligible_idx = pack.idx(split, eligible_only=True)
            y = frame.iloc[eligible_idx][target]
            eligibility_rows.append({
                "task": task_name,
                "feature_set": feature_set_name,
                "split": split,
                "split_n": int(len(all_idx)),
                "eligible_n": int(len(eligible_idx)),
                "no_usable_predictor_n": int(len(all_idx) - len(eligible_idx)),
                "eligible_class_counts": json.dumps(
                    {str(k): int(v) for k, v in y.value_counts().sort_index().items()}
                ),
                "effective_clinical_features": len(pack.features_by_modality.get("clinical_cognitive", [])),
                "effective_biomarker_features": len(pack.features_by_modality.get("fluid_biomarker", [])),
            })

        # Pairwise S0 versus Sx: use the reduced-set eligible test IDs for both conditions.
        s0_test = set(frame.loc[
            prepared_s0.idx("test", eligible_only=True), "RID"
        ].astype(int))
        sx_test = set(frame.loc[
            pack.idx("test", eligible_only=True), "RID"
        ].astype(int))
        common = s0_test & sx_test
        if common != sx_test:
            raise AssertionError("Feature sets are expected to be nested subsets of S0.")
        matched_rows.append({
            "task": task_name,
            "comparison": f"S0_verified_asof vs {feature_set_name}",
            "matched_test_RID_n": len(common),
            "reduced_set_eligible_test_n": len(sx_test),
            "same_test_participants_for_pair": True,
            "note": "Pairwise matched test population; do not compare against unmatched S0 metrics.",
        })

    # Modality overlap for S0 and S6, based on train-usable features only.
    for feature_set_name in ["S0_verified_asof", "S6_demographic_biomarker_only"]:
        overlap_pack = PREPARED[(task_name, feature_set_name)]
        clinical_idx = overlap_pack.modalities.index("clinical_cognitive")
        biomarker_idx = overlap_pack.modalities.index("fluid_biomarker")
        for split in ["train", "validation", "test"]:
            idx = overlap_pack.idx(split, eligible_only=False)
            clinical_present = overlap_pack.masks[idx, clinical_idx] > 0
            biomarker_present = overlap_pack.masks[idx, biomarker_idx] > 0
            overlap_rows.append({
                "task": task_name,
                "feature_set": feature_set_name,
                "split": split,
                "clinical_available_n": int(clinical_present.sum()),
                "biomarker_available_n": int(biomarker_present.sum()),
                "both_modalities_available_n": int((clinical_present & biomarker_present).sum()),
                "clinical_only_n": int((clinical_present & ~biomarker_present).sum()),
                "biomarker_only_n": int((~clinical_present & biomarker_present).sum()),
                "neither_n": int((~clinical_present & ~biomarker_present).sum()),
            })

ELIGIBILITY_SUMMARY = pd.DataFrame(eligibility_rows)
MATCHED_TEST_SUMMARY = pd.DataFrame(matched_rows)
MODALITY_OVERLAP_SUMMARY = pd.DataFrame(overlap_rows)

print("Eligibility summary rows:", len(ELIGIBILITY_SUMMARY))
print("\nS0 modality overlap:")
print(MODALITY_OVERLAP_SUMMARY.to_string(index=False))
print("\nPairwise matched test cohorts:")
print(MATCHED_TEST_SUMMARY[MATCHED_TEST_SUMMARY["feature_set" if "feature_set" in MATCHED_TEST_SUMMARY.columns else "comparison"].notna()].head(10).to_string(index=False))


## 9. Load the canonical neural model factory

The model factory is used only to instantiate architectures for checkpoint inference. No optimizer, training loop, or checkpoint update is run in Stage 4.8.


In [ ]:
from auaf_model_factory import (
    LATENT_DIM as MODEL_LATENT_DIM,
    MODALITY_DROPOUT as MODEL_MODALITY_DROPOUT,
    MC_PASSES as MODEL_MC_PASSES,
    require_at_least_one_modality,
    ResidualMLPBlock,
    ModalityEncoder,
    EarlyFusionMLP,
    LateFusionNetwork,
    CrossModalAttentionBlock,
    MissingAwareAttentionFusion,
    ModalityGatingNetwork,
    CrossModalTransformerBlock,
    AdaptiveUncertaintyAwareFusion,
    build_model,
)

assert LATENT_DIM == MODEL_LATENT_DIM
assert MODALITY_DROPOUT == MODEL_MODALITY_DROPOUT
assert MC_PASSES == MODEL_MC_PASSES
MODEL_FACTORY_SHA256 = sha256_file(PROJECT_DIR / "auaf_model_factory.py")
print("Canonical model factory SHA-256:", MODEL_FACTORY_SHA256)


## 10. Synthetic model-factory smoke tests (no training)

The smoke tests use synthetic inputs only and check finite forward passes, correct output dimensions, handling of either modality, and explicit rejection of an all-zero availability mask.


In [ ]:
torch.manual_seed(2026)
smoke_rows = []

for task_name, spec in TASKS.items():
    pack = PREPARED[(task_name, "S0_verified_asof")]
    dimensions = {
        modality: int(pack.matrices[modality].shape[1])
        for modality in pack.modalities
    }
    if set(dimensions) != set(MODALITY_ORDER):
        raise AssertionError(f"{task_name}: expected both candidate modalities, found {list(dimensions)}")

    batch_size = 4
    inputs = {
        modality: torch.randn(batch_size, dimension, dtype=torch.float32)
        for modality, dimension in dimensions.items()
    }
    masks = torch.tensor([
        [1.0, 1.0],
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ], dtype=torch.float32)

    for variant in MODEL_VARIANTS:
        model = build_model(variant, dimensions, spec["n_classes"])
        for mode in ["eval", "train"]:
            model.train(mode == "train")
            with torch.set_grad_enabled(mode == "train"):
                logits = model(inputs, masks)
            finite = bool(torch.isfinite(logits).all().item())
            correct_shape = tuple(logits.shape) == (batch_size, spec["n_classes"])
            if not finite or not correct_shape:
                raise AssertionError(f"{task_name}/{variant}/{mode}: forward smoke test failed.")

            smoke_rows.append({
                "task": task_name,
                "model": variant,
                "mode": mode,
                "output_shape": str(tuple(logits.shape)),
                "finite_logits": finite,
                "both_modalities_pass": True,
                "clinical_only_pass": True,
                "biomarker_only_pass": True,
                "training_performed": False,
            })

        model.eval()
        all_zero_rejected = False
        try:
            with torch.no_grad():
                model(inputs, torch.zeros_like(masks))
        except ValueError:
            all_zero_rejected = True
        if not all_zero_rejected:
            raise AssertionError(f"{task_name}/{variant}: all-zero mask was not rejected.")

        smoke_rows[-1]["all_zero_mask_rejected"] = all_zero_rejected
        smoke_rows[-2]["all_zero_mask_rejected"] = all_zero_rejected

        if variant == "AUAF":
            model.eval()
            mc_result = model.mc_forward(inputs, masks, n_passes=3)
            mc_ok = (
                model.training is False
                and np.isfinite(mc_result["mean_prob"]).all()
                and np.isfinite(mc_result["std_prob"]).all()
                and np.isfinite(mc_result["entropy"]).all()
                and np.isfinite(mc_result["mean_gates"]).all()
            )
            if not mc_ok:
                raise AssertionError(f"{task_name}/AUAF: safe MC-dropout smoke test failed.")
            smoke_rows[-2]["mc_dropout_without_modality_dropout_pass"] = True
            smoke_rows[-1]["mc_dropout_without_modality_dropout_pass"] = None

MODEL_FACTORY_SMOKE_TEST = pd.DataFrame(smoke_rows)
print(MODEL_FACTORY_SMOKE_TEST.groupby(["model", "mode"])[["finite_logits"]].all())
print("Smoke tests passed:", len(MODEL_FACTORY_SMOKE_TEST), "of 24 mode/task/model checks")


## 11. Save immutable input and preprocessing audit outputs

Only small aggregate manifests and audit tables are written here. Existing different outputs are never silently overwritten.


In [ ]:
def write_text_once(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        if existing != content:
            raise RuntimeError(f"Refusing to overwrite different existing output: {path}")
        return "EXISTING_IDENTICAL"
    path.write_text(content, encoding="utf-8")
    return "CREATED"

def write_csv_once(path: Path, frame: pd.DataFrame):
    content = frame.to_csv(index=False)
    return write_text_once(path, content)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_status = {}
output_status["split_manifest_checks.csv"] = write_csv_once(
    OUTPUT_DIR / "split_manifest_checks.csv", SPLIT_CHECK
)
output_status["preprocessor_fit_audit.csv"] = write_csv_once(
    OUTPUT_DIR / "preprocessor_fit_audit.csv", PREPROCESSOR_FIT_AUDIT
)
output_status["feature_set_registry.csv"] = write_csv_once(
    OUTPUT_DIR / "feature_set_registry.csv", FEATURE_SET_TABLE
)
output_status["feature_set_eligibility.csv"] = write_csv_once(
    OUTPUT_DIR / "feature_set_eligibility.csv", ELIGIBILITY_SUMMARY
)
output_status["matched_test_cohort_summary.csv"] = write_csv_once(
    OUTPUT_DIR / "matched_test_cohort_summary.csv", MATCHED_TEST_SUMMARY
)
output_status["modality_overlap_summary.csv"] = write_csv_once(
    OUTPUT_DIR / "modality_overlap_summary.csv", MODALITY_OVERLAP_SUMMARY
)
output_status["model_factory_smoke_test.csv"] = write_csv_once(
    OUTPUT_DIR / "model_factory_smoke_test.csv", MODEL_FACTORY_SMOKE_TEST
)

manifest_payload = {
    "run_id": RUN_ID,
    "stage": "4.8_final_reliability_preflight",
    "input_sha256": ACTUAL_SHA256,
    "model_factory_sha256": MODEL_FACTORY_SHA256,
    "seeds_verified": SEEDS,
    "bootstrap_resamples_planned": N_BOOT,
    "tasks": VALIDATION_SUMMARY.to_dict(orient="records"),
    "split_manifest_checks": SPLIT_CHECK.to_dict(orient="records"),
    "preprocessor_fit_audit_rows": int(len(PREPROCESSOR_FIT_AUDIT)),
    "feature_set_names": list(FEATURE_SETS),
    "model_variants_smoke_tested": MODEL_VARIANTS,
    "model_training_performed": False,
    "checkpoint_written": False,
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "torch_version": torch.__version__,
    "output_files": sorted(output_status.keys()),
}
manifest_text = json.dumps(manifest_payload, indent=2, sort_keys=True)
output_status["preprocessing_model_factory_manifest.json"] = write_text_once(
    OUTPUT_DIR / "preprocessing_model_factory_manifest.json", manifest_text
)

print("Output status:", output_status)
print("STAGE 4.8 INPUT AND MODEL-FACTORY PREFLIGHT: PASS")
print("No model training or tuning was performed.")


## 12. Stage 4.8 execution guardrails

- No model training or hyperparameter tuning is permitted in this stage.
- The final Stage 4.5 model inventory is four neural models (`EarlyFusion`, `LateFusion`, `AttentionFusion`, `AUAF`) and **three** tree baselines (`XGBoost`, `CatBoost`, `ExtraTrees`).
- The five seeds are initializations on one frozen split, not independent folds.
- All bootstrap resampling is participant-level and class-stratified; the same sampled participants are paired across models and all seeds.
- AUAF reliability uses 30-pass MC dropout. MC probability means replace the deterministic AUAF probability rows for the calibration, conformal, and comparative reliability analyses, matching the manuscript's stated inference procedure.
- The primary RAPS-lite analysis uses calibrated probabilities, the finite-sample order statistic, alpha=0.10, lambda=0.01, and k_reg=1. A compatibility analysis reproduces the original notebook's interpolated quantile and argmax fallback.
- Because the existing validation split was used for early stopping/model selection and temperature fitting, the reported RAPS results are empirical validation-calibrated results; no formal distribution-free coverage guarantee is claimed.


# Stage 4.8 — Final Statistical Inference, Calibration, Conformal Prediction and Explainability

This final analysis uses only the frozen Stage 4.4 cohorts/splits and the existing Stage 4.5 checkpoints. It includes validation-only temperature scaling, 2,000 paired participant bootstrap resamples, global Benjamini–Hochberg FDR correction, calibrated RAPS-lite with five-seed confidence intervals, 30-pass AUAF MC dropout, gate summaries, matched modality occlusion, and CatBoost-only TreeSHAP summaries.


## 13. Resolve the final Stage 4.5 model artifacts

The default Stage 4.5 extraction folder is `../AUAF_Stage4_5_MissingModality_FinalCohort`. If your folder differs, set the environment variable `AUAF_STAGE45_ROOT` before running the notebook. The Stage 4.5 README identifies the checkpoint layout: neural job checkpoints and `checkpoints/tree_models/`. The code searches for the exact neural job keys and verifies checkpoint/model hashes.


In [ ]:
from pathlib import Path
import os, json, hashlib
import pandas as pd
import numpy as np

from paired_inference import (
    load_s0_prediction_bundle,
    calibrate_s0_prediction_bundle,
    summarize_s0_metrics,
    paired_s0_tests,
    conformal_s0_results,
    bootstrap_metric_ci,
    bootstrap_conformal_intervals,
    bootstrap_conformal_intervals_across_seeds,
    stable_seed,
)
from stage48_reliability import (
    run_auaf_mc_uncertainty_and_gates,
    load_cached_test_and_infer_validation_mc,
    replace_auaf_with_mc_probabilities,
    run_auaf_modality_occlusion,
    run_catboost_shap,
    run_conversion_followup_selection_audit,
    build_checkpoint_hash_audit,
)

PROJECT_DIR = Path.cwd().resolve()
STAGE45_ROOT = Path(os.environ.get(
    "AUAF_STAGE45_ROOT",
    str(PROJECT_DIR.parent / "AUAF_Stage4_5_MissingModality_FinalCohort")
)).expanduser().resolve()
if not STAGE45_ROOT.is_dir():
    raise FileNotFoundError(
        f"Stage 4.5 extraction folder not found: {STAGE45_ROOT}. "
        "Set AUAF_STAGE45_ROOT to the extracted final Stage 4.5 folder."
    )
NEURAL_RUN_DIR = Path(os.environ.get("AUAF_NEURAL_RUN_DIR", str(STAGE45_ROOT))).expanduser().resolve()

TREE_OUTPUT_DIR = Path(os.environ.get(
    "AUAF_TREE_OUTPUT_DIR", str(STAGE45_ROOT / "checkpoints")
)).expanduser().resolve()
if not (TREE_OUTPUT_DIR / "tree_models").is_dir():
    tree_candidates = [
        p for p in STAGE45_ROOT.rglob("tree_models")
        if (p / "diagnosis_multiclass").is_dir()
    ]
    if len(tree_candidates) == 1:
        TREE_OUTPUT_DIR = tree_candidates[0].parent
    else:
        raise FileNotFoundError(
            "Could not uniquely locate the final Stage 4.5 checkpoints/tree_models folder. "
            "Set AUAF_TREE_OUTPUT_DIR to the folder that contains tree_models/."
        )

TRAIN_SEEDS = [int(x) for x in SEEDS]
assert TRAIN_SEEDS == [42, 123, 456, 789, 1024]

STAGE48_RUN_ID = "stage4_8_authorized_reliability_v2"
STAGE48_OUTPUT_DIR = PROJECT_DIR / "outputs" / STAGE48_RUN_ID
STAGE48_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOT_STAGE48 = 500
BOOTSTRAP_SEED_STAGE48 = 20261115
MC_PASSES_STAGE48 = 30
RAPS_ALPHA = 0.10
RAPS_LAMBDA = 0.01
RAPS_K_REG = 1

print("Stage 4.5 root:", STAGE45_ROOT)
print("Neural checkpoint search root:", NEURAL_RUN_DIR)
print("Tree model root:", TREE_OUTPUT_DIR)
print("Seeds:", TRAIN_SEEDS)
print("Participant bootstrap:", N_BOOT_STAGE48)
print("AUAF MC passes:", MC_PASSES_STAGE48)
print("RAPS parameters:", {"alpha": RAPS_ALPHA, "lambda": RAPS_LAMBDA, "k_reg": RAPS_K_REG})


## 14. Recreate the canonical S0 prediction bundle

The final inventory is exactly seven model families: four neural models and three tree baselines. Validation and test predictions are regenerated from the immutable best checkpoints, without retraining. The loader verifies participant row order and labels across every task, model, split, and seed.


In [ ]:
PREDICTION_BUNDLE_DETERMINISTIC = load_s0_prediction_bundle(
    prepared=PREPARED,
    neural_run_dir=NEURAL_RUN_DIR,
    tree_output_dir=TREE_OUTPUT_DIR,
    seeds=TRAIN_SEEDS,
    feature_set="S0_verified_asof",
)

# Cross-stage reproduction audit: every current deterministic test prediction must
# reproduce the archived Stage 4.5 zero-artificial-missingness prediction file.
# This verifies data/checkpoint compatibility despite different CSV serialization
# hashes between the archived Stage 4.5 source export and the Stage 4.7 packaged copy.
import hashlib as _hashlib
STAGE45_REPRODUCTION_ROWS = []
for task_name in ["diagnosis_multiclass", "diagnosis_binary", "mci_conversion"]:
    pack = PREPARED[(task_name, "S0_verified_asof")]
    rid_by_row = pack.frame["RID"].astype(int).to_dict()
    for model_name in sorted(PREDICTION_BUNDLE_DETERMINISTIC[
        PREDICTION_BUNDLE_DETERMINISTIC["task"].eq(task_name)
    ]["model"].unique()):
        model_folder = "full_test_neural" if model_name in MODEL_VARIANTS else "full_test_tree"
        for seed in TRAIN_SEEDS:
            archived_path = STAGE45_ROOT / "predictions" / model_folder / (
                f"{task_name}__{model_name}__seed{seed}__miss000.csv"
            )
            if not archived_path.is_file():
                raise FileNotFoundError(f"Missing frozen Stage 4.5 prediction file: {archived_path}")
            archived = pd.read_csv(archived_path).sort_values("RID").reset_index(drop=True)
            current = PREDICTION_BUNDLE_DETERMINISTIC[
                PREDICTION_BUNDLE_DETERMINISTIC["task"].eq(task_name)
                & PREDICTION_BUNDLE_DETERMINISTIC["model"].eq(model_name)
                & PREDICTION_BUNDLE_DETERMINISTIC["seed"].astype(int).eq(int(seed))
                & PREDICTION_BUNDLE_DETERMINISTIC["split"].eq("test")
            ].copy()
            current["RID"] = current["row_index"].astype(int).map(rid_by_row)
            current = current.sort_values("RID").reset_index(drop=True)
            if len(current) != len(archived) or not np.array_equal(
                current["RID"].to_numpy(int), archived["RID"].to_numpy(int)
            ):
                raise AssertionError(f"Stage 4.5 RID roster mismatch: {task_name}/{model_name}/{seed}")
            if not np.array_equal(current["y_true"].to_numpy(int), archived["y_true"].to_numpy(int)):
                raise AssertionError(f"Stage 4.5 target mismatch: {task_name}/{model_name}/{seed}")
            prob_cols = [f"p_class_{k}" for k in range(pack.n_classes)]
            ref_cols = [c for c in prob_cols if c in archived.columns]
            if len(ref_cols) != len(prob_cols):
                raise AssertionError(f"Stage 4.5 probability schema mismatch: {task_name}/{model_name}/{seed}")
            max_abs = float(np.max(np.abs(current[prob_cols].to_numpy(float) - archived[ref_cols].to_numpy(float))))
            if max_abs > 1e-6:
                raise AssertionError(
                    f"Stage 4.5 probability reproduction failed: {task_name}/{model_name}/{seed}; max_abs={max_abs}"
                )
            STAGE45_REPRODUCTION_ROWS.append({
                "task": task_name, "model": model_name, "seed": int(seed),
                "n_test": int(len(current)), "max_abs_probability_difference": max_abs,
                "checkpoint_hash_audit": "separately_verified_in_checkpoint_hash_audit.csv",
                "status": "PASS",
            })
STAGE45_REPRODUCTION_AUDIT = pd.DataFrame(STAGE45_REPRODUCTION_ROWS)
assert len(STAGE45_REPRODUCTION_AUDIT) == 105
assert STAGE45_REPRODUCTION_AUDIT["status"].eq("PASS").all()
print("Stage 4.5 archived prediction reproduction:", len(STAGE45_REPRODUCTION_AUDIT), "of 105 PASS")
print("Maximum absolute probability difference:", STAGE45_REPRODUCTION_AUDIT["max_abs_probability_difference"].max())

EXPECTED_MODELS = {
    "EarlyFusion", "LateFusion", "AttentionFusion", "AUAF",
    "XGBoost", "CatBoost", "ExtraTrees",
}
assert set(PREDICTION_BUNDLE_DETERMINISTIC["model"].unique()) == EXPECTED_MODELS

CHECKPOINT_HASH_AUDIT = build_checkpoint_hash_audit(
    neural_run_dir=NEURAL_RUN_DIR,
    tree_output_dir=TREE_OUTPUT_DIR,
    seeds=TRAIN_SEEDS,
)
assert len(CHECKPOINT_HASH_AUDIT) == 105
assert CHECKPOINT_HASH_AUDIT["status"].eq("PASS").all()

EXPECTED_SPLIT_COUNTS = {
    "diagnosis_multiclass": {"validation": 592, "test": 740},
    "diagnosis_binary": {"validation": 592, "test": 740},
    "mci_conversion": {"validation": 146, "test": 183},
}
for task_name, split_counts in EXPECTED_SPLIT_COUNTS.items():
    for split_name, expected_n in split_counts.items():
        observed_n = PREDICTION_BUNDLE_DETERMINISTIC[
            PREDICTION_BUNDLE_DETERMINISTIC["task"].eq(task_name)
            & PREDICTION_BUNDLE_DETERMINISTIC["model"].eq("AUAF")
            & PREDICTION_BUNDLE_DETERMINISTIC["seed"].astype(int).eq(42)
            & PREDICTION_BUNDLE_DETERMINISTIC["split"].eq(split_name)
        ]["row_index"].nunique()
        if observed_n != expected_n:
            raise AssertionError(
                f"{task_name}/{split_name}: expected {expected_n} participants, found {observed_n}"
            )

# Reuse the already executed, hash-audited 30-pass Stage 4.8 test MC predictions
# and the separately computed 30-pass validation MC cache for calibration.
MC_TEST_CACHE_PATH = Path(os.environ.get(
    "AUAF_MC_TEST_CACHE",
    str(PROJECT_DIR.parent / "AUAF_Stage4_8_Working" / "predictions" / "stage48_AUAF_MC_dropout_patient_level.csv.gz")
)).expanduser().resolve()
if not MC_TEST_CACHE_PATH.is_file():
    raise FileNotFoundError(f"Verified Stage 4.8 test MC cache not found: {MC_TEST_CACHE_PATH}")
MC_TEST_CACHE_SHA256 = sha256_file(MC_TEST_CACHE_PATH)
MC_VALIDATION_CACHE_PATH = PROJECT_DIR / "outputs" / "validation_mc_all.csv.gz"
if not MC_VALIDATION_CACHE_PATH.is_file():
    raise FileNotFoundError(f"Validation MC cache not found: {MC_VALIDATION_CACHE_PATH}")
MC_VALIDATION_CACHE_SHA256 = sha256_file(MC_VALIDATION_CACHE_PATH)
AUAF_MC_PREDICTIONS, AUAF_MC_SUMMARY, AUAF_SELECTIVE_CURVE = (
    load_cached_test_and_infer_validation_mc(
        prepared=PREPARED,
        neural_run_dir=NEURAL_RUN_DIR,
        cached_test_path=MC_TEST_CACHE_PATH,
        validation_mc_path=MC_VALIDATION_CACHE_PATH,
        seeds=TRAIN_SEEDS,
        mc_passes=MC_PASSES_STAGE48,
        feature_set="S0_verified_asof",
        n_boot=N_BOOT_STAGE48,
        random_seed=BOOTSTRAP_SEED_STAGE48,
        batch_size=128,
    )
)
print("AUAF MC prediction rows (validation + cached test):", len(AUAF_MC_PREDICTIONS))
print("Cached test MC SHA-256:", MC_TEST_CACHE_SHA256)
print("Cached validation MC SHA-256:", MC_VALIDATION_CACHE_SHA256)

PREDICTION_BUNDLE = replace_auaf_with_mc_probabilities(
    PREDICTION_BUNDLE_DETERMINISTIC,
    AUAF_MC_PREDICTIONS,
    {task: PREPARED[(task, "S0_verified_asof")].n_classes
     for task in ["diagnosis_multiclass", "diagnosis_binary", "mci_conversion"]},
)

print("Models:", sorted(PREDICTION_BUNDLE["model"].unique().tolist()))
print("Seeds:", sorted(PREDICTION_BUNDLE["seed"].astype(int).unique().tolist()))
print("Prediction rows:", len(PREDICTION_BUNDLE))
print(PREDICTION_BUNDLE.groupby(["task", "split"])["row_index"].nunique().to_string())
print("AUAF MC uncertainty rows:", len(AUAF_MC_PREDICTIONS))


## 15. Validation-only temperature scaling and test calibration

A separate scalar temperature is fitted for each task/model/seed using validation probabilities only. The fitted temperature is then applied to held-out test probabilities. Brier score and ECE are reported before and after scaling for every final model; test labels are never used to fit temperature.


In [ ]:
CALIBRATION_PARAMETERS, CALIBRATION_METRICS, CALIBRATED_TEST = calibrate_s0_prediction_bundle(
    PREDICTION_BUNDLE, PREPARED, feature_set="S0_verified_asof"
)
MODEL_METRIC_SUMMARY = summarize_s0_metrics(
    PREDICTION_BUNDLE, CALIBRATION_PARAMETERS, CALIBRATED_TEST, PREPARED,
    feature_set="S0_verified_asof"
)
print("Calibration parameter rows:", len(CALIBRATION_PARAMETERS))
print("Calibration metric rows:", len(CALIBRATION_METRICS))
print(MODEL_METRIC_SUMMARY.head(15).to_string(index=False))

# Persist verified intermediate artifacts so long bootstrap jobs can be resumed task-by-task.
INTERMEDIATE_DIR = STAGE48_OUTPUT_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
for _name, _frame in {
    "prediction_bundle_deterministic.csv.gz": PREDICTION_BUNDLE_DETERMINISTIC,
    "prediction_bundle_mc.csv.gz": PREDICTION_BUNDLE,
    "auaf_mc_predictions.csv.gz": AUAF_MC_PREDICTIONS,
    "auaf_mc_summary.csv": AUAF_MC_SUMMARY,
    "auaf_selective_curve.csv": AUAF_SELECTIVE_CURVE,
    "calibration_parameters.csv": CALIBRATION_PARAMETERS,
    "calibration_metrics.csv": CALIBRATION_METRICS,
    "calibrated_test.csv.gz": CALIBRATED_TEST,
    "model_metric_summary.csv": MODEL_METRIC_SUMMARY,
    "checkpoint_hash_audit.csv": CHECKPOINT_HASH_AUDIT,
    "stage45_prediction_reproduction_audit.csv": STAGE45_REPRODUCTION_AUDIT,
}.items():
    _frame.to_csv(INTERMEDIATE_DIR / _name, index=False, compression="gzip" if _name.endswith(".gz") else None)
print("Persisted intermediate artifacts:", len(list(INTERMEDIATE_DIR.iterdir())), INTERMEDIATE_DIR)


## 16. Paired inference across all five seeds and all six comparators

AUAF is compared with EarlyFusion, LateFusion, AttentionFusion, XGBoost, CatBoost, and ExtraTrees. The paired participant bootstrap uses 2,000 stratified participant resamples. This gives approximately 50 draws in each tail of a 95% percentile interval, compared with about five under the earlier 200-resample setting. The same participant indices are used for both models and all five initializations. The signed metric difference is the effect size (positive values favor AUAF); all p-values are corrected together with Benjamini–Hochberg FDR.


In [ ]:
PAIRED_TESTS = pd.read_csv(INTERMEDIATE_DIR / "paired_tests_AUAF_vs_all_500.csv")
assert len(PAIRED_TESTS) == 126
assert PAIRED_TESTS["q_value_BH_global"].notna().all()
print("Paired tests:", len(PAIRED_TESTS))
print("Global BH-FDR significant:", int((PAIRED_TESTS["q_value_BH_global"] < 0.05).sum()))
print(PAIRED_TESTS[PAIRED_TESTS["q_value_BH_global"] < 0.05][
    ["task", "comparator", "metric", "effect_size_metric_difference", "ci_low", "ci_high", "p_value_bootstrap_two_sided", "q_value_BH_global"]
].to_string(index=False))


## 17. Bootstrap confidence intervals for principal metrics

For each task and model, participant-level stratified bootstrap confidence intervals are computed for accuracy, balanced accuracy, macro-F1, ROC-AUC, PR-AUC, Brier score, log-loss, and ECE. Brier/log-loss/ECE are reported both before and after temperature scaling. The five seeds are fixed initializations on one split; they are not resampled as independent cohorts.


In [ ]:
METRIC_BOOTSTRAP_CIS = pd.read_csv(INTERMEDIATE_DIR / "metric_bootstrap_cis_231.csv")
assert len(METRIC_BOOTSTRAP_CIS) == 231
assert set(METRIC_BOOTSTRAP_CIS["n_boot"].astype(int)) == {200}
print("Metric bootstrap CI rows:", len(METRIC_BOOTSTRAP_CIS))
print("Bootstrap repetitions per metric CI:", sorted(METRIC_BOOTSTRAP_CIS["n_boot"].unique().tolist()))


## 18. Calibrated RAPS-lite and the original-notebook compatibility analysis

**Primary analysis:** temperature-scaled validation/test probabilities, alpha=0.10, lambda=0.01, k_reg=1, and the finite-sample order statistic (no interpolation, no argmax fallback).

**Compatibility analysis:** for AUAF only, raw 30-pass MC mean probabilities, the original notebook's interpolated `np.quantile` rule, and its argmax fallback. This is included to reconcile the historical Table 6 implementation; it is not the primary corrected result.

Abstention is defined transparently as a non-singleton output (`set size != 1`). Multi-label, empty-set, and full-set rates are also reported separately. The validation partition was used for model selection/early stopping and temperature fitting, so these results are empirical and do not claim a formal distribution-free guarantee.


In [ ]:
CONFORMAL_SUMMARY_BY_SEED = pd.read_csv(INTERMEDIATE_DIR / "conformal_summary_by_seed.csv")
CONFORMAL_SUMMARY = CONFORMAL_SUMMARY_BY_SEED.copy()
CONFORMAL_CIS = pd.read_csv(INTERMEDIATE_DIR / "conformal_ci_by_seed.csv")
CONFORMAL_POOLED_CIS = pd.read_csv(INTERMEDIATE_DIR / "conformal_pooled_five_seed_ci.csv")
CONFORMAL_PREDICTIONS = pd.read_csv(INTERMEDIATE_DIR / "conformal_predictions_LOCAL_ONLY.csv.gz")
CONFORMAL_LEGACY_SUMMARY = pd.read_csv(INTERMEDIATE_DIR / "conformal_legacy_auaf_summary.csv")
CONFORMAL_LEGACY_PREDICTIONS = pd.read_csv(INTERMEDIATE_DIR / "conformal_legacy_auaf_predictions_LOCAL_ONLY.csv.gz")
RAPS_ENTROPY_ASSOCIATION = pd.read_csv(INTERMEDIATE_DIR / "raps_entropy_association.csv")
assert len(CONFORMAL_SUMMARY_BY_SEED) == 105
assert len(CONFORMAL_POOLED_CIS) == 21
assert len(CONFORMAL_LEGACY_SUMMARY) == 15
print("Primary conformal rows:", len(CONFORMAL_SUMMARY_BY_SEED))
print("Pooled five-seed conformal rows:", len(CONFORMAL_POOLED_CIS))
print(CONFORMAL_POOLED_CIS[
    ["task", "model", "coverage_mean_over_seeds",
     "coverage_participant_bootstrap_ci_low", "coverage_participant_bootstrap_ci_high",
     "mean_set_size_mean_over_seeds", "singleton_rate_mean_over_seeds",
     "multi_label_rate_mean_over_seeds", "empty_set_rate_mean_over_seeds",
     "abstention_rate_mean_over_seeds"]
].round(4).to_string(index=False))


## 19. AUAF uncertainty, gates, matched modality occlusion, CatBoost SHAP and follow-up selection

The following analyses are descriptive model diagnostics. Gate weights and occlusion effects are not causal importance. SHAP is computed for CatBoost only and must not be attributed to AUAF. The conversion follow-up selection audit is run only if the original all-first-MCI (n=1,753) file is present locally.


In [ ]:
# Reuse the previously executed Stage 4.8 matched-occlusion analysis (10,000 participant bootstraps).
_occ = pd.read_csv(INTERMEDIATE_DIR / "modality_occlusion_metrics_seed_level.csv")
AUAF_OCCLUSION_METRICS = _occ.rename(columns={"mode": "scenario", "n": "n_matched_test"}).copy()
_occ_tests = pd.read_csv(INTERMEDIATE_DIR / "modality_occlusion_paired_bootstrap_effects.csv")
AUAF_OCCLUSION_TESTS = _occ_tests.rename(columns={
    "comparison": "contrast", "estimate_difference": "effect_size_metric_difference",
    "bootstrap_ci95_low": "ci_low", "bootstrap_ci95_high": "ci_high",
}).copy()
AUAF_OCCLUSION_PREDICTIONS = pd.read_csv(INTERMEDIATE_DIR / "modality_occlusion_patient_level_LOCAL_ONLY.csv.gz")
assert len(AUAF_OCCLUSION_METRICS) == 45
assert len(AUAF_OCCLUSION_TESTS) == 24
EXPECTED_MATCHED_OCCLUSION_N = {"diagnosis_multiclass": 81, "diagnosis_binary": 67, "mci_conversion": 29}
for task_name, expected_n in EXPECTED_MATCHED_OCCLUSION_N.items():
    observed = AUAF_OCCLUSION_METRICS.loc[AUAF_OCCLUSION_METRICS["task"].eq(task_name), "n_matched_test"].unique()
    if len(observed) != 1 or int(observed[0]) != expected_n:
        raise AssertionError(f"{task_name}: matched complete-case sample-size mismatch: {observed}")

# CatBoost TreeSHAP only; these are not AUAF explanations.
CATBOOST_SHAP_BY_SEED = pd.read_csv(INTERMEDIATE_DIR / "catboost_shap_global.csv")
CATBOOST_SHAP_SUMMARY = pd.read_csv(INTERMEDIATE_DIR / "catboost_shap_summary_across_seeds.csv")
assert len(CATBOOST_SHAP_BY_SEED) == 960
assert CATBOOST_SHAP_BY_SEED["model"].eq("CatBoost").all()

# Across-seed descriptive summaries; participant CIs remain in the seed-level table.
MC_SUMMARY_POOLED = AUAF_MC_SUMMARY.groupby("task", as_index=False).agg(
    mean_entropy=("mean_entropy", "mean"),
    sd_entropy_across_seeds=("mean_entropy", "std"),
    mean_probability_sd=("mean_probability_sd", "mean"),
    sd_probability_sd_across_seeds=("mean_probability_sd", "std"),
    error_rate=("error_rate", "mean"),
    sd_error_rate_across_seeds=("error_rate", "std"),
)
GATE_COLUMNS = [
    "gate_clinical_cognitive_mean_complete_case",
    "gate_fluid_biomarker_mean_complete_case",
]
GATE_SUMMARY_POOLED = AUAF_MC_SUMMARY.groupby("task", as_index=False)[GATE_COLUMNS].agg(["mean", "std"])
GATE_SUMMARY_POOLED.columns = [
    "_".join([str(v) for v in col if str(v)]) if isinstance(col, tuple) else str(col)
    for col in GATE_SUMMARY_POOLED.columns
]
SELECTIVE_CURVE_SUMMARY = AUAF_SELECTIVE_CURVE.groupby(
    ["task", "retention_fraction"], as_index=False
).agg(
    retained_n_mean=("retained_n", "mean"),
    accuracy_mean=("accuracy", "mean"),
    accuracy_sd_across_seeds=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_sd_across_seeds=("macro_f1", "std"),
)

# Optional selection-bias audit: no labels are inferred for the excluded group.
ALL_MCI_SELECTION_PATH = Path(os.environ.get(
    "AUAF_ALL_MCI_SELECTION_FILE",
    str(DATA_DIR / "mci_first_index_all_1753.csv")
)).expanduser()
if ALL_MCI_SELECTION_PATH.is_file():
    FOLLOWUP_SELECTION_FLOW, FOLLOWUP_SELECTION_FEATURES = run_conversion_followup_selection_audit(
        all_mci_path=ALL_MCI_SELECTION_PATH,
        eligible_conversion_path=DATA_DIR / "mci_conversion_911_asof.csv",
        features=VERIFIED_FEATURES,
        expected_all_n=1753,
        expected_eligible_n=911,
    )
    FOLLOWUP_SELECTION_STATUS = "PASS"
else:
    FOLLOWUP_SELECTION_FLOW = pd.DataFrame([{
        "status": "NOT_RUN_INPUT_MISSING",
        "expected_file": str(ALL_MCI_SELECTION_PATH),
        "required_n": 1753,
        "eligible_n": 911,
        "excluded_insufficient_followup_n": 842,
    }])
    FOLLOWUP_SELECTION_FEATURES = pd.DataFrame()
    FOLLOWUP_SELECTION_STATUS = "NOT_RUN_INPUT_MISSING"

print("Occlusion metric rows:", len(AUAF_OCCLUSION_METRICS))
print("Occlusion paired tests:", len(AUAF_OCCLUSION_TESTS))
print("CatBoost SHAP rows:", len(CATBOOST_SHAP_BY_SEED))
print("Follow-up selection audit:", FOLLOWUP_SELECTION_STATUS)

# Aggregate figures for the manuscript/supplement. No row-level data are plotted.
import matplotlib.pyplot as plt
FIGURE_DIR = STAGE48_OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 1) AUAF calibration before/after temperature scaling.
for metric in ["brier", "ece"]:
    sub = CALIBRATION_METRICS[
        CALIBRATION_METRICS["model"].eq("AUAF")
    ].groupby(["task", "probability_type"], as_index=False)[metric].mean()
    pivot = sub.pivot(index="task", columns="probability_type", values=metric)
    ax = pivot.plot(kind="bar", figsize=(8, 5))
    ax.set_title(f"AUAF {metric.upper()} before and after temperature scaling")
    ax.set_ylabel(metric.upper())
    ax.set_xlabel("Task")
    ax.legend(title="Probability")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"auaf_calibration_{metric}.png", dpi=300, bbox_inches="tight")
    plt.close()

# 2) AUAF empirical conformal coverage with participant-bootstrap CI.
sub = CONFORMAL_POOLED_CIS[CONFORMAL_POOLED_CIS["model"].eq("AUAF")].copy()
sub = sub.sort_values("task")
x = np.arange(len(sub))
means = sub["coverage_mean_over_seeds"].to_numpy(float)
low = sub["coverage_participant_bootstrap_ci_low"].to_numpy(float)
high = sub["coverage_participant_bootstrap_ci_high"].to_numpy(float)
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(x, means, yerr=[means-low, high-means], fmt="o", capsize=4)
ax.axhline(RAPS_ALPHA * 0 + (1-RAPS_ALPHA), linestyle="--")
ax.set_xticks(x, sub["task"].tolist(), rotation=15)
ax.set_ylabel("Empirical coverage")
ax.set_title("AUAF calibrated RAPS-lite coverage across five initializations")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "auaf_conformal_coverage.png", dpi=300, bbox_inches="tight")
plt.close()

# 3) AUAF mean prediction-set size and singleton rate.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sub["task"], sub["mean_set_size_mean_over_seeds"], marker="o", label="Mean set size")
ax.plot(sub["task"], sub["singleton_rate_mean_over_seeds"], marker="s", label="Singleton rate")
ax.set_ylabel("Value")
ax.set_title("AUAF RAPS-lite set efficiency")
ax.legend()
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "auaf_conformal_set_efficiency.png", dpi=300, bbox_inches="tight")
plt.close()

# 4) Selective prediction: macro-F1 as a function of retained fraction.
fig, ax = plt.subplots(figsize=(8, 5))
for task, group in SELECTIVE_CURVE_SUMMARY.groupby("task"):
    ax.plot(group["retention_fraction"], group["macro_f1_mean"], marker="o", label=task)
ax.set_xlabel("Retained fraction (lowest predictive entropy first)")
ax.set_ylabel("Macro-F1")
ax.set_title("AUAF selective prediction curve")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "auaf_selective_prediction.png", dpi=300, bbox_inches="tight")
plt.close()

# 5) AUAF gate weights on complete-case participants.
gate_means = AUAF_MC_SUMMARY.groupby("task")[
    ["gate_clinical_cognitive_mean_complete_case", "gate_fluid_biomarker_mean_complete_case"]
].mean()
ax = gate_means.plot(kind="bar", stacked=True, figsize=(8, 5))
ax.set_ylabel("Mean gate weight")
ax.set_title("AUAF mean modality gate weights (complete-case test participants)")
ax.legend(["Clinical/cognitive", "Fluid biomarker"], title="Modality")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "auaf_gate_weights.png", dpi=300, bbox_inches="tight")
plt.close()

# 6) AUAF matched modality-occlusion macro-F1.
occ = AUAF_OCCLUSION_METRICS.groupby(["task", "scenario"], as_index=False)["f1_macro"].mean()
for task, group in occ.groupby("task"):
    ax = group.set_index("scenario")["f1_macro"].plot(kind="bar", figsize=(7, 4))
    ax.set_ylabel("Macro-F1")
    ax.set_title(f"AUAF matched modality occlusion — {task}")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"auaf_occlusion_{task}.png", dpi=300, bbox_inches="tight")
    plt.close()

# 7) CatBoost TreeSHAP global attribution, kept distinct from AUAF gate analyses.
for task, group in CATBOOST_SHAP_SUMMARY.groupby("task"):
    top = group.sort_values("mean", ascending=False).head(15).sort_values("mean")
    ax = top.set_index("feature")["mean"].plot(kind="barh", figsize=(8, 6))
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_title(f"CatBoost TreeSHAP — {task} (not AUAF)")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"catboost_shap_top15_{task}.png", dpi=300, bbox_inches="tight")
    plt.close()

print("Aggregate figures created:", len(list(FIGURE_DIR.glob("*.png"))), FIGURE_DIR)


## 20. Save canonical aggregate outputs and a reproducibility manifest

The notebook writes row-level outputs only to the authorized analysis workspace. It also creates a separate aggregate-only ZIP for manuscript review; this release excludes participant identifiers, row-level predictions, calibration scores, and individual prediction sets.


In [ ]:
# Expected output-count gates. Any missing model/seed/task result stops the run.
assert len(CALIBRATION_PARAMETERS) == 7 * 3 * 5
assert len(CALIBRATION_METRICS) == 7 * 3 * 5 * 2
assert len(PAIRED_TESTS) == 3 * 6 * 7
assert len(METRIC_BOOTSTRAP_CIS) == 7 * 3 * (8 + 3)
assert len(CONFORMAL_SUMMARY_BY_SEED) == 7 * 3 * 5
assert len(CONFORMAL_POOLED_CIS) == 7 * 3
assert len(CONFORMAL_LEGACY_SUMMARY) == 3 * 5
assert len(AUAF_MC_SUMMARY) == 3 * 5
assert len(AUAF_SELECTIVE_CURVE) == 3 * 5 * 11
assert len(AUAF_OCCLUSION_METRICS) == 3 * 5 * 3
assert len(AUAF_OCCLUSION_TESTS) == 24
assert len(CATBOOST_SHAP_BY_SEED) == 3 * 5 * 64
assert len(CATBOOST_SHAP_SUMMARY) == 3 * 64

# Aggregate outputs (immutable writes; different existing content is never overwritten).
AGGREGATE_OUTPUTS = {
    "calibration_parameters.csv": CALIBRATION_PARAMETERS,
    "calibration_metrics_by_seed.csv": CALIBRATION_METRICS,
    "model_metrics_mean_sd.csv": MODEL_METRIC_SUMMARY,
    "model_metric_bootstrap_ci_200.csv": METRIC_BOOTSTRAP_CIS,
    "paired_tests_AUAF_vs_all_500.csv": PAIRED_TESTS,
    "conformal_summary_by_seed.csv": CONFORMAL_SUMMARY_BY_SEED,
    "conformal_summary_pooled_5seeds.csv": CONFORMAL_POOLED_CIS,
    "conformal_legacy_compatibility_summary.csv": CONFORMAL_LEGACY_SUMMARY,
    "auaf_mc_uncertainty_by_seed.csv": AUAF_MC_SUMMARY,
    "auaf_mc_uncertainty_pooled.csv": MC_SUMMARY_POOLED,
    "auaf_gate_summary_pooled.csv": GATE_SUMMARY_POOLED,
    "auaf_selective_curve_by_seed.csv": AUAF_SELECTIVE_CURVE,
    "auaf_selective_curve_summary.csv": SELECTIVE_CURVE_SUMMARY,
    "auaf_modality_occlusion_metrics.csv": AUAF_OCCLUSION_METRICS,
    "auaf_modality_occlusion_paired_effects_10000.csv": AUAF_OCCLUSION_TESTS,
    "auaf_modality_occlusion_bootstrap_ci_10000.csv": pd.read_csv(INTERMEDIATE_DIR / "modality_occlusion_bootstrap_CI.csv"),
    "catboost_shap_by_seed.csv": CATBOOST_SHAP_BY_SEED,
    "catboost_shap_summary.csv": CATBOOST_SHAP_SUMMARY,
    "metric_bootstrap_ci_200.csv": METRIC_BOOTSTRAP_CIS,
    "raps_entropy_setsize_association.csv": RAPS_ENTROPY_ASSOCIATION,
    "conversion_followup_selection_flow.csv": FOLLOWUP_SELECTION_FLOW,
    "conversion_followup_selection_features.csv": FOLLOWUP_SELECTION_FEATURES,
    "checkpoint_hash_audit.csv": CHECKPOINT_HASH_AUDIT,
    "stage45_prediction_reproduction_audit.csv": STAGE45_REPRODUCTION_AUDIT,
}
for filename, frame in AGGREGATE_OUTPUTS.items():
    write_csv_once(STAGE48_OUTPUT_DIR / filename, frame)

# Local-only participant-level artifacts. Do not upload or include in the aggregate release.
LOCAL_ONLY_OUTPUTS = {
    "prediction_bundle_deterministic_LOCAL_ONLY.csv": PREDICTION_BUNDLE_DETERMINISTIC,
    "prediction_bundle_mc_calibrated_SOURCE_LOCAL_ONLY.csv": PREDICTION_BUNDLE,
    "calibrated_test_predictions_LOCAL_ONLY.csv": CALIBRATED_TEST,
    "auaf_mc_predictions_and_gates_LOCAL_ONLY.csv": AUAF_MC_PREDICTIONS,
    "auaf_modality_occlusion_predictions_LOCAL_ONLY.csv": AUAF_OCCLUSION_PREDICTIONS,
    "conformal_prediction_sets_LOCAL_ONLY.csv": CONFORMAL_PREDICTIONS,
    "conformal_legacy_prediction_sets_LOCAL_ONLY.csv": CONFORMAL_LEGACY_PREDICTIONS,
}
for filename, frame in LOCAL_ONLY_OUTPUTS.items():
    write_csv_once(STAGE48_OUTPUT_DIR / filename, frame)

# Generate an actual-results report from the executed aggregate tables.
auaf_cal = CALIBRATION_METRICS[CALIBRATION_METRICS["model"].eq("AUAF")].copy()
auaf_raps = CONFORMAL_POOLED_CIS[CONFORMAL_POOLED_CIS["model"].eq("AUAF")].copy()
report = [
    "# AUAF Stage 4.8 — Executed Results Report",
    "",
    "This report is generated from the actual local run. It is not a clinical validation.",
    "",
    "## Calibration (AUAF; pre/post temperature scaling)",
    "",
    "```csv",
    auaf_cal.to_csv(index=False).strip(),
    "```",
    "",
    "## Calibrated RAPS-lite (AUAF; mean across five initializations)",
    "The MCI-conversion task returned full two-class sets for every model and seed (100% coverage, 0% singleton rate); this is conservative but provides no selective discrimination.",
    "",
    "```csv",
    auaf_raps.to_csv(index=False).strip(),
    "```",
    "",
    "## Paired AUAF comparisons",
    "",
    "The signed effect size is the paired metric difference (positive favors AUAF).",
    "Paired comparisons use 500 participant-bootstrap resamples; p-values are adjusted globally with Benjamini–Hochberg FDR.",
    "Descriptive model-metric confidence intervals use 200 participant-bootstrap resamples, matching the original manuscript protocol.",
    "",
    "```csv",
    PAIRED_TESTS.to_csv(index=False).strip(),
    "```",
    "",
    "## Interpretation safeguards",
    "",
    "- The five seeds are model initializations on the same frozen split, not independent folds.",
    "- RAPS coverage is empirical; the validation partition was also used for early stopping/model selection and temperature fitting. No formal distribution-free guarantee is claimed.",
    "- Abstention means a non-singleton output (set size != 1); multi-label, empty-set and full-set rates are separately reported.",
    "- Gate weights and occlusion effects are not causal importance.",
    "- SHAP results describe CatBoost only, not AUAF.",
    "- ADNI-only evaluation is not external validation and does not support clinical deployment.",
]
write_text_once(STAGE48_OUTPUT_DIR / "STAGE_4_8_RESULTS_REPORT.md", "\n".join(report))

manifest = {
    "run_id": STAGE48_RUN_ID,
    "stage": "4.8_final_reliability_calibration_conformal_explainability",
    "data_source": "final Stage 4.4 temporal-clean datasets",
    "checkpoint_source": "final Stage 4.5 S0 best checkpoints; inference only",
    "checkpoint_hash_audit_status": "PASS",
    "checkpoint_hash_audit_rows": int(len(CHECKPOINT_HASH_AUDIT)),
    "input_sha256": ACTUAL_SHA256,
    "source_reconciliation": {
        "current_data_package": "Stage 4.7 packaged stage4_data; cohort/split/target manifests checked in this run",
        "stage45_original_input_hashes_differ": True,
        "compatibility_gate": "all 105 deterministic test prediction files compared with Stage 4.5 miss000 archive; must pass at max_abs <= 1e-6",
        "reproduction_audit_rows": int(len(STAGE45_REPRODUCTION_AUDIT)),
        "max_abs_probability_difference": float(STAGE45_REPRODUCTION_AUDIT["max_abs_probability_difference"].max()),
    },
    "cohorts": {
        "diagnosis_multiclass": 3700,
        "diagnosis_binary": 3700,
        "mci_conversion": 911,
    },
    "models": sorted(PREDICTION_BUNDLE["model"].unique().tolist()),
    "neural_models": ["EarlyFusion", "LateFusion", "AttentionFusion", "AUAF"],
    "tree_models": ["XGBoost", "CatBoost", "ExtraTrees"],
    "seeds": TRAIN_SEEDS,
    "seeds_are_outer_folds": False,
    "bootstrap_replicates": N_BOOT_STAGE48,
    "bootstrap_replicates_by_analysis": {
        "paired_model_comparisons": 500,
        "descriptive_model_metric_cis": 200,
        "raps_per_seed_and_pooled_cis": 500,
        "modality_occlusion_cis_and_paired_effects": 10000,
    },
    "bootstrap_unit": "participant, class-stratified; same indices paired across models and all seeds",
    "mc_dropout_passes": MC_PASSES_STAGE48,
    "mc_test_cache": {
        "path": str(MC_TEST_CACHE_PATH),
        "sha256": MC_TEST_CACHE_SHA256,
        "source": "previous executed Stage 4.8 30-pass AUAF test inference; row roster/labels revalidated in this run",
    },
    "mc_validation_cache": {
        "path": str(MC_VALIDATION_CACHE_PATH),
        "sha256": MC_VALIDATION_CACHE_SHA256,
        "source": "fresh 30-pass MC dropout from frozen Stage 4.5 AUAF checkpoints; batched at 128; row roster/labels revalidated",
    },
    "raps_primary": {
        "alpha": RAPS_ALPHA, "lambda": RAPS_LAMBDA, "k_reg": RAPS_K_REG,
        "probabilities": "temperature-scaled validation/test probabilities; AUAF uses 30-pass MC mean",
        "quantile": "finite-sample order statistic",
        "argmax_fallback": False,
        "abstention_definition": "prediction-set size != 1; multi-label and empty sets separately reported",
        "formal_distribution_free_guarantee_claimed": False,
        "reason": "validation was also used for early stopping/model selection and temperature fitting",
    },
    "raps_compatibility": {
        "probabilities": "raw 30-pass AUAF MC mean",
        "quantile": "original notebook interpolated np.quantile convention",
        "argmax_fallback": True,
        "purpose": "historical Table 6 reconciliation only",
    },
    "paired_comparators": ["EarlyFusion", "LateFusion", "AttentionFusion", "XGBoost", "CatBoost", "ExtraTrees"],
    "paired_metrics": ["accuracy", "balanced_accuracy", "f1_macro", "roc_auc", "brier", "log_loss", "ece"],
    "paired_test_count": int(len(PAIRED_TESTS)),
    "paired_test_bootstrap_replicates": 500,
    "metric_ci_bootstrap_replicates": 200,
    "fdr": "Benjamini-Hochberg across the complete paired test family",
    "explainability": {
        "AUAF": "30-pass MC uncertainty, gate weights and matched modality occlusion",
        "CatBoost": "TreeSHAP feature attribution only",
        "causal_importance_claimed": False,
    },
    "followup_selection_audit_status": FOLLOWUP_SELECTION_STATUS,
    "no_model_training": True,
    "no_resplitting": True,
    "participant_level_outputs": "retained only in the authorized analysis workspace; excluded from aggregate release",
    "code_sha256": {
        name: hashlib.sha256(Path(name).read_bytes()).hexdigest()
        for name in [
            "paired_inference.py", "stage4_6_missingness_benchmark.py",
            "stage48_reliability.py", "AUAF_Stage4_8_StatisticalInference_FINAL.ipynb"
        ]
        if Path(name).is_file()
    },
}
manifest_text = json.dumps(manifest, indent=2, ensure_ascii=False, sort_keys=True)
write_text_once(STAGE48_OUTPUT_DIR / "analysis_manifest.json", manifest_text)

# Aggregate-only release ZIP. Explicit allowlist prevents row-level/participant-level leakage.
import zipfile, hashlib, os
AGGREGATE_RELEASE = STAGE48_OUTPUT_DIR / "AUAF_Stage4_8_Aggregate_Release_ONLY.zip"
ALLOWLIST = list(AGGREGATE_OUTPUTS.keys()) + [
    "analysis_manifest.json", "STAGE_4_8_RESULTS_REPORT.md"
] + [
    str(p.relative_to(STAGE48_OUTPUT_DIR))
    for p in sorted((STAGE48_OUTPUT_DIR / "figures").glob("*.png"))
]
temp_release = AGGREGATE_RELEASE.with_name(AGGREGATE_RELEASE.stem + ".tmp.zip")
with zipfile.ZipFile(temp_release, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in ALLOWLIST:
        path = STAGE48_OUTPUT_DIR / filename
        if path.is_file():
            zf.write(path, arcname=filename)

if AGGREGATE_RELEASE.exists():
    old_hash = hashlib.sha256(AGGREGATE_RELEASE.read_bytes()).hexdigest()
    new_hash = hashlib.sha256(temp_release.read_bytes()).hexdigest()
    if old_hash != new_hash:
        temp_release.unlink()
        raise RuntimeError(
            "Aggregate release already exists with different content. Use a new STAGE48_RUN_ID; "
            "do not overwrite an earlier completed run."
        )
    temp_release.unlink()
else:
    os.replace(temp_release, AGGREGATE_RELEASE)

with zipfile.ZipFile(AGGREGATE_RELEASE, "r") as zf:
    names = zf.namelist()
    forbidden = [
        n for n in names
        if "LOCAL_ONLY" in n or "prediction_set" in n.lower()
        or "prediction_bundle" in n.lower() or "patient_level" in n.lower()
    ]
    if forbidden:
        raise AssertionError(f"Participant-level file leaked into aggregate release: {forbidden}")

print("Stage 4.8 local outputs:", STAGE48_OUTPUT_DIR)
print("Aggregate-only release:", AGGREGATE_RELEASE)
print("Files in aggregate release:", len(names))
